## Interactive PCD clustering comparison app



Import libraries

In [27]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import DBSCAN, KMeans, SpectralClustering
import hdbscan

In [28]:


def _pcd_bytes_to_xyz(raw_bytes):
    raw_bytes = bytes(raw_bytes)
    lines = raw_bytes.decode("utf-8", errors="ignore").splitlines()
    fields = None
    data_start = None
    data_mode = None
    for index, line in enumerate(lines):
        parts = line.strip().split()
        if not parts or parts[0].startswith("#"):
            continue
        if parts[0].upper() == "FIELDS":
            fields = parts[1:]
        elif parts[0].upper() == "DATA":
            data_mode = parts[1].lower() if len(parts) > 1 else None
            data_start = index + 1
            break
    if data_start is None or data_mode != "ascii":
        raise ValueError("This app supports ASCII PCD files only.")
    rows = [[float(value) for value in line.replace(",", " ").split()] for line in lines[data_start:] if line.strip() and not line.strip().startswith("#")]
    data = np.asarray(rows, dtype=float)
    if data.size == 0:
        raise ValueError("The PCD file contains no point data.")
    if data.ndim == 1:
        data = data.reshape(1, -1)
    columns = [fields.index(axis) for axis in ("x", "y", "z")] if fields and all(axis in fields for axis in ("x", "y", "z")) else [0, 1, 2]
    if data.shape[1] < 3:
        raise ValueError("The PCD file must contain x, y and z data.")
    return data[:, columns]



In [37]:
def load_pcd(file_path):
    """ Open pcd file from file path and return list of points """
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        header = []
        data_type = None
        fields = None

        for line in f:
            stripped = line.strip()
            if not stripped or stripped.startswith("#"):
                continue
            header.append(stripped)
            if stripped.startswith("DATA"):
                parts = stripped.split()
                if len(parts) < 2:
                    raise ValueError("Invalid PCD DATA line")
                data_type = parts[1].lower()
                break
            if stripped.startswith("FIELDS"):
                fields = stripped.split()[1:]

        if data_type != "ascii":
            raise ValueError("Only ASCII PCD files are supported")

        rows = []
        for line in f:
            stripped = line.strip()
            if not stripped or stripped.startswith("#"):
                continue
            values = stripped.replace(",", " ").split()
            if values:
                rows.append([float(value) for value in values])

        if not rows:
            raise ValueError("No point data found in PCD file")

        data = np.array(rows, dtype=float)
        if data.ndim == 1:
            data = data.reshape(1, -1)

        if fields is not None:
            field_map = {name: idx for idx, name in enumerate(fields)}
            if all(key in field_map for key in ("x", "y", "z")):
                points = data[:, [field_map["x"], field_map["y"], field_map["z"]]]
            else:
                points = data[:, :3]
        else:
            points = data[:, :3]

        return points


In [30]:


def _box_mask(points_array, box):
    lower = np.array([box["xmin"], box["ymin"], box["zmin"]])
    upper = np.array([box["xmax"], box["ymax"], box["zmax"]])
    return np.all((points_array >= lower) & (points_array <= upper), axis=1)



In [31]:

def _best_cluster(labels_array, manual_mask_array):
    candidates = [label for label in np.unique(labels_array) if label != -1] or list(np.unique(labels_array))
    best = None
    for label in candidates:
        predicted = labels_array == label
        tp = int(np.count_nonzero(predicted & manual_mask_array))
        fp = int(np.count_nonzero(predicted & ~manual_mask_array))
        fn = int(np.count_nonzero(~predicted & manual_mask_array))
        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-12)
        candidate = {"label": label, "mask": predicted, "tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall, "f1": f1}
        if best is None or candidate["f1"] > best["f1"]:
            best = candidate
    return best



# Initialise app

In [ ]:
app_state = {"raw_points": None, "points": None, "boxes": [], "results": {}}
upload = widgets.FileUpload(accept=".pcd", multiple=False, description="Upload PCD")
status = widgets.HTML(value="Upload an ASCII PCD file to begin.")
box_output = widgets.Output()
box_preview_output = widgets.Output()
xmin = widgets.FloatText(description="xmin", layout=widgets.Layout(width="145px"))
xmax = widgets.FloatText(description="xmax", layout=widgets.Layout(width="145px"))
ymin = widgets.FloatText(description="ymin", layout=widgets.Layout(width="145px"))
ymax = widgets.FloatText(description="ymax", layout=widgets.Layout(width="145px"))
zmin = widgets.FloatText(description="zmin", layout=widgets.Layout(width="145px"))
zmax = widgets.FloatText(description="zmax", layout=widgets.Layout(width="145px"))
add_box = widgets.Button(description="Add box", button_style="success", icon="plus")
clear_boxes = widgets.Button(description="Clear boxes", button_style="warning", icon="trash")
keep_fraction = widgets.FloatSlider(
    value=1.0,
    min=0.01,
    max=1.0,
    step=0.01,
    description="Keep points",
    readout_format=".0%",
    continuous_update=False,
)
filter_threshold = widgets.FloatSlider(
    value=1.0,
    min=0.01,
    max=1.0,
    step=0.01,
    description="Distance limit",
    readout_format=".0%",
    continuous_update=False,
)
PREVIEW_POINT_LIMIT = 20000


def subsample(points, keep_fraction):
    if not 0 < keep_fraction <= 1:
        raise ValueError("keep_fraction must be between 0 and 1")
    keep_count = max(1, int(len(points) * keep_fraction))
    indices = np.linspace(0, len(points) - 1, keep_count, dtype=int)
    return points[indices]


def filter(points, threshold):
    distances = np.linalg.norm(points, axis=1)
    threshold_number = threshold * distances.max()
    return points[distances <= threshold_number]


def _box_mask(points_array, box):
    lower = np.array([box["xmin"], box["ymin"], box["zmin"]])
    upper = np.array([box["xmax"], box["ymax"], box["zmax"]])
    return np.all((points_array >= lower) & (points_array <= upper), axis=1)


def _box_summary():
    if not app_state["boxes"]:
        return "No manual boxes added."
    return "<br>".join(
        f"Box {index}: x [{box['xmin']:.3g}, {box['xmax']:.3g}], "
        f"y [{box['ymin']:.3g}, {box['ymax']:.3g}], "
        f"z [{box['zmin']:.3g}, {box['zmax']:.3g}]"
        for index, box in enumerate(app_state["boxes"], 1)
    )


def _set_status(message, color="#444"):
    status.value = f'<span style="color:{color}">{message}</span>'


def _draw_preview_box(axes, box, color):
    x_values = [box["xmin"], box["xmax"]]
    y_values = [box["ymin"], box["ymax"]]
    z_values = [box["zmin"], box["zmax"]]
    for x_value in x_values:
        for y_value in y_values:
            axes.plot([x_value, x_value], [y_value, y_value], z_values, color=color)
    for x_value in x_values:
        for z_value in z_values:
            axes.plot([x_value, x_value], y_values, [z_value, z_value], color=color)
    for y_value in y_values:
        for z_value in z_values:
            axes.plot(x_values, [y_value, y_value], [z_value, z_value], color=color)


def _render_box_preview():
    with box_output:
        clear_output(wait=True)
        display(widgets.HTML(value=_box_summary()))
    with box_preview_output:
        clear_output(wait=True)
        points = app_state["points"]
        if points is None or not len(points):
            display(widgets.HTML(value="Upload a point cloud to preview bounding boxes."))
            return

        preview_indices = np.linspace(
            0, len(points) - 1, min(len(points), PREVIEW_POINT_LIMIT), dtype=int
        )
        preview_points = points[preview_indices]
        point_classes = np.zeros(len(preview_points), dtype=int)
        for class_index, box in enumerate(app_state["boxes"], 1):
            selected = _box_mask(preview_points, box) & (point_classes == 0)
            point_classes[selected] = class_index

        figure = plt.figure(figsize=(10, 7))
        axes = figure.add_subplot(111, projection="3d")
        background = point_classes == 0
        axes.scatter(*preview_points[background].T, c="lightgray", s=2, alpha=0.25, label="Other points")
        palette = plt.get_cmap("tab10")
        for class_index, box in enumerate(app_state["boxes"], 1):
            selected = point_classes == class_index
            color = palette((class_index - 1) % 10)
            axes.scatter(
                *preview_points[selected].T,
                color=color,
                s=5,
                label=f"Box {class_index} points",
            )
            _draw_preview_box(axes, box, color)
        axes.set_title(f"Bounding-box preview ({len(points):,} points, up to {len(preview_points):,} plotted)")
        axes.set_xlabel("X")
        axes.set_ylabel("Y")
        axes.set_zlabel("Z")
        axes.set_box_aspect(np.maximum(np.ptp(preview_points, axis=0), 1e-6))
        if app_state["boxes"]:
            axes.legend(loc="upper right")
        figure.tight_layout()
        plt.show()
        plt.close(figure)


def _update_bounds(points):
    minimum = points.min(axis=0)
    maximum = points.max(axis=0)
    for field, value in zip((xmin, ymin, zmin), minimum):
        field.value = float(value)
    for field, value in zip((xmax, ymax, zmax), maximum):
        field.value = float(value)


def _apply_point_settings(_=None):
    if app_state["raw_points"] is None:
        return
    filtered_points = filter(app_state["raw_points"], filter_threshold.value)
    app_state["points"] = subsample(filtered_points, keep_fraction.value)
    app_state["boxes"] = []
    app_state["results"] = {}
    _update_bounds(app_state["points"])
    _set_status(
        f"Using {len(app_state['points']):,} of {len(app_state['raw_points']):,} uploaded points. "
        "Filtering resets boxes; add them again on the preview.",
        "#176b3a",
    )
    _render_box_preview()


def _on_upload(change):
    if not upload.value:
        return
    uploaded = next(iter(upload.value.values())) if isinstance(upload.value, dict) else upload.value[0]
    try:
        app_state["raw_points"] = _pcd_bytes_to_xyz(uploaded["content"])
        app_state["boxes"] = []
        app_state["results"] = {}
        _apply_point_settings()
        _set_status(
            f"Loaded {len(app_state['raw_points']):,} points from {uploaded['name']}; "
            f"using {len(app_state['points']):,} after filtering/subsampling.",
            "#176b3a",
        )
    except Exception as error:
        app_state["raw_points"] = None
        app_state["points"] = None
        _set_status(str(error), "#a11")
        _render_box_preview()


def _on_add_box(_):
    if app_state["points"] is None:
        _set_status("Upload a PCD file before adding a box.", "#a11")
        return
    box = {
        "xmin": xmin.value,
        "xmax": xmax.value,
        "ymin": ymin.value,
        "ymax": ymax.value,
        "zmin": zmin.value,
        "zmax": zmax.value,
    }
    if any(box[f"{axis}min"] > box[f"{axis}max"] for axis in ("x", "y", "z")):
        _set_status("Each minimum must be less than or equal to its maximum.", "#a11")
        return
    app_state["boxes"].append(box)
    app_state["results"] = {}
    _set_status(f"Added box {len(app_state['boxes'])}; adjust the coordinates to test another box.", "#176b3a")
    _render_box_preview()


def _on_clear_boxes(_):
    app_state["boxes"] = []
    app_state["results"] = {}
    _set_status("Cleared all bounding boxes.")
    _render_box_preview()


upload.observe(_on_upload, names="value")
keep_fraction.observe(_apply_point_settings, names="value")
filter_threshold.observe(_apply_point_settings, names="value")
add_box.on_click(_on_add_box)
clear_boxes.on_click(_on_clear_boxes)
display(widgets.VBox([
    upload,
    status,
    widgets.HTML("<b>Point filtering and subsampling</b>"),
    filter_threshold,
    keep_fraction,
    widgets.HTML("<b>Bounding boxes</b>"),
    widgets.HBox([xmin, xmax]),
    widgets.HBox([ymin, ymax]),
    widgets.HBox([zmin, zmax]),
    widgets.HBox([add_box, clear_boxes]),
    box_output,
    box_preview_output,
]))
print("Upload and bounding-box preview ready.")

In [33]:


def _box_summary():
    if not app_state["boxes"]:
        return "No manual boxes added."
    return "<br>".join(f"Box {i}: x [{b['xmin']:.3g}, {b['xmax']:.3g}], y [{b['ymin']:.3g}, {b['ymax']:.3g}], z [{b['zmin']:.3g}, {b['zmax']:.3g}]" for i, b in enumerate(app_state["boxes"], 1))


def _set_status(message, color="#444"):
    status.value = f'<span style="color:{color}">{message}</span>'


def _on_upload(change):
    if not upload.value:
        return
    uploaded = next(iter(upload.value.values()))
    try:
        app_state["points"] = load_pcd(uploaded["content"])
        app_state["boxes"] = []
        app_state["results"] = {}
        minimum = app_state["points"].min(axis=0)
        maximum = app_state["points"].max(axis=0)
        for field, value in zip((xmin, ymin, zmin), minimum):
            field.value = float(value)
        for field, value in zip((xmax, ymax, zmax), maximum):
            field.value = float(value)
        _set_status(f"Loaded {len(app_state['points']):,} points from {uploaded['name']}.", "#176b3a")
        with box_output:
            clear_output(wait=True)
            display(widgets.HTML(value=_box_summary()))
    except Exception as error:
        app_state["points"] = None
        _set_status(str(error), "#a11")


def _on_add_box(_):
    if app_state["points"] is None:
        _set_status("Upload a PCD file before adding a box.", "#a11")
        return
    box = {"xmin": xmin.value, "xmax": xmax.value, "ymin": ymin.value, "ymax": ymax.value, "zmin": zmin.value, "zmax": zmax.value}
    if any(box[f"min_{axis}"] > box[f"max_{axis}"] for axis in ("x", "y", "z")):
        _set_status("Each minimum must be less than or equal to its maximum.", "#a11")
        return
    app_state["boxes"].append(box)
    with box_output:
        clear_output(wait=True)
        display(widgets.HTML(value=_box_summary()))


def _on_clear_boxes(_):
    app_state["boxes"] = []
    app_state["results"] = {}
    with box_output:
        clear_output(wait=True)
        display(widgets.HTML(value=_box_summary()))
    with plot_output:
        clear_output(wait=True)
    with metrics_output:
        clear_output(wait=True)


upload.observe(_on_upload, names="value")
add_box.on_click(_on_add_box)
clear_boxes.on_click(_on_clear_boxes)

In [34]:
def _on_add_box(_):
    if app_state["points"] is None:
        _set_status("Upload a PCD file before adding a box.", "#a11")
        return
    box = {"xmin": xmin.value, "xmax": xmax.value, "ymin": ymin.value, "ymax": ymax.value, "zmin": zmin.value, "zmax": zmax.value}
    if any(box[f"{axis}min"] > box[f"{axis}max"] for axis in ("x", "y", "z")):
        _set_status("Each minimum must be less than or equal to its maximum.", "#a11")
        return
    app_state["boxes"].append(box)
    with box_output:
        clear_output(wait=True)
        display(widgets.HTML(value=_box_summary()))


add_box.on_click(_on_add_box)


def _cluster_metrics(predicted, actual):
    true_positive = int(np.count_nonzero(predicted & actual))
    false_positive = int(np.count_nonzero(predicted & ~actual))
    false_negative = int(np.count_nonzero(~predicted & actual))
    precision = true_positive / max(true_positive + false_positive, 1)
    recall = true_positive / max(true_positive + false_negative, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    return {
        "tp": true_positive,
        "fp": false_positive,
        "fn": false_negative,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def _best_clusters(labels_array, manual_masks):
    cluster_labels = [label for label in np.unique(labels_array) if label != -1]
    class_results = [{
        "label": None,
        "mask": np.zeros(len(labels_array), dtype=bool),
        **_cluster_metrics(np.zeros(len(labels_array), dtype=bool), manual_mask),
    } for manual_mask in manual_masks]
    if not cluster_labels or not manual_masks:
        return class_results

    scores = np.zeros((len(manual_masks), len(cluster_labels)))
    for class_index, manual_mask in enumerate(manual_masks):
        for cluster_index, label in enumerate(cluster_labels):
            scores[class_index, cluster_index] = _cluster_metrics(
                labels_array == label, manual_mask
            )["f1"]

    class_indices, cluster_indices = linear_sum_assignment(-scores)
    for class_index, cluster_index in zip(class_indices, cluster_indices):
        label = cluster_labels[cluster_index]
        predicted = labels_array == label
        class_results[class_index] = {
            "label": label,
            "mask": predicted,
            **_cluster_metrics(predicted, manual_masks[class_index]),
        }
    return class_results


def _grid_search(points_array, manual_masks, candidates):
    best = None
    for parameters, estimator in candidates:
        try:
            labels = estimator.fit_predict(points_array)
            class_results = _best_clusters(labels, manual_masks)
            box_macro_f1 = float(np.mean([result["f1"] for result in class_results]))
            candidate = {
                "params": parameters,
                "labels": labels,
                "classes": class_results,
                "box_macro_f1": box_macro_f1,
            }
            if best is None or candidate["box_macro_f1"] > best["box_macro_f1"]:
                best = candidate
        except Exception:
            continue
    if best is None:
        raise ValueError("No valid parameter combination produced a clustering.")
    return best


def _format_parameters(parameters):
    return ", ".join(f"{name}={value:g}" if isinstance(value, float) else f"{name}={value}" for name, value in parameters.items())


def _multiclass_predictions(result):
    predicted_labels = np.zeros(len(result["labels"]), dtype=int)
    for class_index, class_result in enumerate(result["classes"], 1):
        if class_result["label"] is not None:
            predicted_labels[result["labels"] == class_result["label"]] = class_index
    return predicted_labels


def _classification_metrics(true_labels, predicted_labels, class_count):
    confusion = np.zeros((class_count, class_count), dtype=int)
    np.add.at(confusion, (true_labels, predicted_labels), 1)
    class_metrics = []
    for class_index in range(class_count):
        true_positive = int(confusion[class_index, class_index])
        false_positive = int(confusion[:, class_index].sum() - true_positive)
        false_negative = int(confusion[class_index, :].sum() - true_positive)
        support = int(confusion[class_index, :].sum())
        precision = true_positive / max(true_positive + false_positive, 1)
        recall = true_positive / max(support, 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-12)
        class_metrics.append({
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "support": support,
        })

    supports = np.array([item["support"] for item in class_metrics], dtype=float)
    total_support = max(float(supports.sum()), 1.0)
    macro = {
        metric: float(np.mean([item[metric] for item in class_metrics]))
        for metric in ("precision", "recall", "f1")
    }
    weighted = {
        metric: float(np.average([item[metric] for item in class_metrics], weights=supports))
        if supports.sum() else 0.0
        for metric in ("precision", "recall", "f1")
    }
    accuracy = float(np.count_nonzero(true_labels == predicted_labels) / total_support)
    return class_metrics, macro, weighted, accuracy, confusion


def _render_results(points_array, true_labels):
    class_count = len(app_state["boxes"]) + 1
    class_names = ["Background"] + [f"Box {index}" for index in range(1, class_count)]
    reports = {}
    for name, result in app_state["results"].items():
        predicted_labels = _multiclass_predictions(result)
        reports[name] = _classification_metrics(true_labels, predicted_labels, class_count)

    with plot_output:
        clear_output(wait=True)
        figure, axes = plt.subplots(2, 2, figsize=(14, 10), subplot_kw={"projection": "3d"})
        palette = plt.get_cmap("tab10")
        for axis, (name, result) in zip(axes.flat, app_state["results"].items()):
            predicted_labels = _multiclass_predictions(result)
            background = predicted_labels == 0
            axis.scatter(points_array[background, 0], points_array[background, 1], points_array[background, 2], c="k", s=2, alpha=0.2)
            for class_index in range(1, class_count):
                selected = predicted_labels == class_index
                axis.scatter(
                    points_array[selected, 0],
                    points_array[selected, 1],
                    points_array[selected, 2],
                    color=palette((class_index - 1) % 10),
                    s=5,
                    label=f"{class_names[class_index]} (cluster {result['classes'][class_index - 1]['label']})",
                )
            axis.set_title(f"{name}: macro F1 {reports[name][1]['f1']:.3f}")
            axis.set_xlabel("X")
            axis.set_ylabel("Y")
            axis.set_zlabel("Z")
            if class_count > 1:
                axis.legend(fontsize=8)
        figure.tight_layout()
        plt.show()

    with metrics_output:
        clear_output(wait=True)
        report_headers = ["Algorithm", "Class", "Best parameters", "Best cluster", "Precision", "Recall", "F1", "Support"]
        report_rows = []
        for name, result in app_state["results"].items():
            class_metrics, macro, weighted, accuracy, confusion = reports[name]
            report_rows.append([name, "Background", _format_parameters(result["params"]), "-", *(f"{class_metrics[0][metric]:.3f}" for metric in ("precision", "recall", "f1")), class_metrics[0]["support"]])
            for class_index, class_result in enumerate(result["classes"], 1):
                metrics = class_metrics[class_index]
                report_rows.append([name, class_names[class_index], _format_parameters(result["params"]), class_result["label"], *(f"{metrics[metric]:.3f}" for metric in ("precision", "recall", "f1")), metrics["support"]])
            report_rows.append([name, "Macro avg", "-", "-", *(f"{macro[metric]:.3f}" for metric in ("precision", "recall", "f1")), sum(item["support"] for item in class_metrics)])
            report_rows.append([name, "Weighted avg", "-", "-", *(f"{weighted[metric]:.3f}" for metric in ("precision", "recall", "f1")), sum(item["support"] for item in class_metrics)])
            report_rows.append([name, "Accuracy", "-", "-", "-", "-", f"{accuracy:.3f}", sum(item["support"] for item in class_metrics)])

        table = "<table><tr>" + "".join(f"<th style='padding:4px 10px'>{cell}</th>" for cell in report_headers) + "</tr>"
        table += "".join("<tr>" + "".join(f"<td style='padding:4px 10px'>{cell}</td>" for cell in row) + "</tr>" for row in report_rows) + "</table>"
        for name, result in app_state["results"].items():
            confusion = reports[name][4]
            matrix_headers = ["Actual \\ Predicted"] + class_names
            matrix_rows = [[class_names[index], *row.tolist()] for index, row in enumerate(confusion)]
            table += f"<h4>{name} confusion matrix</h4><table><tr>" + "".join(f"<th style='padding:4px 10px'>{cell}</th>" for cell in matrix_headers) + "</tr>"
            table += "".join("<tr>" + "".join(f"<td style='padding:4px 10px'>{cell}</td>" for cell in row) + "</tr>" for row in matrix_rows) + "</table>"
        display(widgets.HTML(value=table))


def _run_algorithms(_):
    points_array = app_state["points"]
    if points_array is None or not app_state["boxes"]:
        _set_status("Upload a PCD and add at least one manual box first.", "#a11")
        return
    true_labels = np.zeros(len(points_array), dtype=int)
    manual_masks = []
    for class_index, box in enumerate(app_state["boxes"], 1):
        mask = _box_mask(points_array, box) & (true_labels == 0)
        manual_masks.append(mask)
        true_labels[mask] = class_index
    try:
        _set_status("Running compact grid searches", "#a11")
        max_clusters = min(n_clusters.value, len(points_array) - 1)
        if max_clusters < 2:
            raise ValueError("At least three points are required for clustering.")
        cluster_values = sorted({
            max(2, min(max_clusters, clusters))
            for clusters in (n_clusters.value - 1, n_clusters.value, n_clusters.value + 1)
        })
        sample_value = max(2, min(min_samples.max, min_samples.value))
        cluster_size_values = sorted({max(2, sample_value // 2), sample_value})
        eps_values = sorted({
            max(1e-6, dbscan_eps.value * factor)
            for factor in (0.8, 1.0, 1.2)
        })
        nearest_neighbors = min(10, max(2, len(points_array) - 1))
        candidates = {
            "K-means": [
                ({"n_clusters": clusters},
                 KMeans(n_clusters=clusters, random_state=42, n_init="auto"))
                for clusters in cluster_values
            ],
            "Spectral": [
                ({"n_clusters": clusters},
                 SpectralClustering(n_clusters=clusters,
                                    affinity="nearest_neighbors",
                                    n_neighbors=nearest_neighbors,
                                    assign_labels="kmeans",
                                    random_state=42))
                for clusters in cluster_values
            ],
            "DBSCAN": [
                ({"eps": eps, "min_samples": sample_value},
                 DBSCAN(eps=eps, min_samples=sample_value))
                for eps in eps_values
            ],
            "HDBSCAN": [
                ({"min_cluster_size": cluster_size, "min_samples": sample_value},
                 hdbscan.HDBSCAN(min_cluster_size=cluster_size,
                                 min_samples=sample_value))
                for cluster_size in cluster_size_values
            ],
        }
        app_state["results"] = {
            name: _grid_search(points_array, manual_masks, parameter_grid)
            for name, parameter_grid in candidates.items()
        }
        _render_results(points_array, true_labels)
        _set_status(
            f"Compared four algorithms against {len(app_state['boxes'])} box classes "
            f"and {int(np.count_nonzero(true_labels == 0)):,} background points. "
            "Overlapping boxes are assigned to the first box added.",
            "#176b3a",
        )
    except Exception as error:
        _set_status(f"Clustering failed: {error}", "#a11")


run_button.on_click(_run_algorithms)

In [35]:
add_box._click_handlers.callbacks = []
add_box.on_click(_on_add_box)
print("Interactive clustering app initialized.")

Interactive clustering app initialized.


In [36]:
upload.unobserve(_on_upload, names="value")


def subsample(points, keep_fraction):
    if not 0 < keep_fraction <= 1:
        raise ValueError("keep_fraction must be between 0 and 1")

    num_keep = max(1, int(len(points) * keep_fraction))
    indices = np.linspace(0, len(points) - 1, num_keep, dtype=int)
    return points[indices]


def filter(points, threshold):
    distances = np.linalg.norm(points, axis=1)
    threshold_number = threshold * distances.max()
    mask = distances <= threshold_number
    return points[mask]


keep_fraction = widgets.FloatSlider(
    value=1.0,
    min=0.01,
    max=1.0,
    step=0.01,
    description="Keep points",
    readout_format=".0%",
    continuous_update=False,
)
filter_threshold = widgets.FloatSlider(
    value=1.0,
    min=0.01,
    max=1.0,
    step=0.01,
    description="Distance limit",
    readout_format=".0%",
    continuous_update=False,
)

app_state["raw_points"] = None


def _update_bounds(points):
    minimum = points.min(axis=0)
    maximum = points.max(axis=0)
    for field, value in zip((xmin, ymin, zmin), minimum):
        field.value = float(value)
    for field, value in zip((xmax, ymax, zmax), maximum):
        field.value = float(value)


def _apply_point_settings(_=None):
    if app_state["raw_points"] is None:
        return
    filtered_points = filter(app_state["raw_points"], filter_threshold.value)
    app_state["points"] = subsample(filtered_points, keep_fraction.value)
    app_state["boxes"] = []
    app_state["results"] = {}
    _update_bounds(app_state["points"])
    _set_status(
        f"Using {len(app_state['points']):,} of {len(app_state['raw_points']):,} uploaded points.",
        "#176b3a",
    )
    with box_output:
        clear_output(wait=True)
        display(widgets.HTML(value=_box_summary()))
    with plot_output:
        clear_output(wait=True)
    with metrics_output:
        clear_output(wait=True)


def _on_upload(change):
    if not upload.value:
        return
    uploaded = next(iter(upload.value.values())) if isinstance(upload.value, dict) else upload.value[0]
    try:
        app_state["raw_points"] = _pcd_bytes_to_xyz(uploaded["content"])
        _apply_point_settings()
        _set_status(
            f"Loaded {len(app_state['raw_points']):,} points from {uploaded['name']}; "
            f"using {len(app_state['points']):,} after filtering/subsampling.",
            "#176b3a",
        )
    except Exception as error:
        app_state["raw_points"] = None
        app_state["points"] = None
        _set_status(str(error), "#a11")


upload.observe(_on_upload, names="value")
keep_fraction.observe(_apply_point_settings, names="value")
filter_threshold.observe(_apply_point_settings, names="value")
display(widgets.VBox([
    upload,
    status,
    widgets.HTML("<b>Point filtering and subsampling</b>"),
    filter_threshold,
    keep_fraction,
    widgets.HTML("<b>Manual bounding boxes</b>"),
    widgets.HBox([xmin, xmax]),
    widgets.HBox([ymin, ymax]),
    widgets.HBox([zmin, zmax]),
    widgets.HBox([add_box, clear_boxes]),
    box_output,
    widgets.HTML("<b>Clustering parameters and grid limits</b>"),
    n_clusters,
    dbscan_eps,
    min_samples,
    run_button,
    metrics_output,
    plot_output,
]))
print("Upload handler and point controls ready.")

Upload handler and point controls ready.
